# **Warung Madura Sales Forecasting**

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

### **Data Loading & Understanding**

In [10]:
sales_dir = 'data/sales.csv'
sales_details_dir = 'data/sales_details.csv'

**Data Loading**

In [11]:
sales_df = pd.read_csv(sales_dir)
sales_details_df = pd.read_csv(sales_details_dir)

In [12]:
sales_df.head()

,order_id,order_date,customer_id,geography_id,payment_method,total_amount
0,O000001,2025-10-25,C0045,G004,QRIS,34350
1,O000002,2025-04-03,C0002,G008,Transfer,66850
2,O000003,2025-07-24,C0042,G005,QRIS,7650
3,O000004,2025-03-14,C0076,G007,Cash,156450
4,O000005,2025-07-19,C0002,G009,Transfer,85500


In [13]:
sales_details_df.head()

,sales_detail_id,order_id,product_id,quantity,unit_price,discount,line_total
0,D0000001,O000001,P014,3,6500,0,19500
1,D0000002,O000001,P010,3,5500,10,14850
2,D0000003,O000002,P007,3,18000,5,51300
3,D0000004,O000002,P017,1,4500,10,4050
4,D0000005,O000002,P011,1,11500,0,11500


**Data Understanding**

In [14]:
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        3000 non-null   str  
 1   order_date      3000 non-null   str  
 2   customer_id     3000 non-null   str  
 3   geography_id    3000 non-null   str  
 4   payment_method  3000 non-null   str  
 5   total_amount    3000 non-null   int64
dtypes: int64(1), str(5)
memory usage: 140.8 KB


In [15]:
sales_details_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7601 entries, 0 to 7600
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   sales_detail_id  7601 non-null   str  
 1   order_id         7601 non-null   str  
 2   product_id       7601 non-null   str  
 3   quantity         7601 non-null   int64
 4   unit_price       7601 non-null   int64
 5   discount         7601 non-null   int64
 6   line_total       7601 non-null   int64
dtypes: int64(4), str(3)
memory usage: 415.8 KB


### **Data Transformation**

**Ubah Tipe Data**

In [16]:
sales_df['order_id'] = sales_df['order_id'].astype(str)
sales_details_df['order_id'] = sales_details_df['order_id'].astype(str)

**Penggabungan Data Order Detail dengan Order untuk Mendapat Konteks Order Date**

In [18]:
sales_details_with_date = sales_details_df.merge(
                                            sales_df[['order_id', 'order_date']],
                                            on  = 'order_id',
                                            how = 'left'
)

In [22]:
sales_details_with_date.head()

,sales_detail_id,order_id,product_id,quantity,unit_price,discount,line_total,order_date
0,D0000001,O000001,P014,3,6500,0,19500,2025-10-25
1,D0000002,O000001,P010,3,5500,10,14850,2025-10-25
2,D0000003,O000002,P007,3,18000,5,51300,2025-04-03
3,D0000004,O000002,P017,1,4500,10,4050,2025-04-03
4,D0000005,O000002,P011,1,11500,0,11500,2025-04-03


**Pembuatan Pivot Table - Jumlah Product Sold tiap Tanggal**

In [21]:
data_for_forecasting = pd.pivot_table(
                                data    = sales_details_with_date,
                                index   = 'order_date',
                                columns = 'product_id',
                                aggfunc = 'sum',
                                values  = 'quantity',
                                fill_value = 0
)

data_for_forecasting = data_for_forecasting.sort_values(by = 'order_date')

In [23]:
data_for_forecasting.head()

product_id,P001,P002,P003,P004,P005,P006,P007,P008,P009,P010,P011,P012,P013,P014,P015,P016,P017,P018,P019,P020
order_date,,,,,,,,,,,,,,,,,,,,
2025-01-01,3,0,1,0,5,1,0,0,0,0,0,0,2,0,0,1,0,4,0,0
2025-01-02,6,4,4,0,8,5,1,8,3,3,0,5,4,4,5,2,5,0,2,5
2025-01-03,0,0,3,3,0,3,2,0,6,1,1,6,1,3,1,0,1,2,3,0
2025-01-04,0,3,0,2,3,2,0,2,2,6,5,2,0,1,0,0,5,0,0,3
2025-01-05,6,1,6,1,0,4,1,1,1,1,1,3,0,0,0,0,4,3,0,2


**Menyimpan data hasil transformasi**

In [25]:
data_for_forecasting.to_csv('data/data_for_forecasting.csv')